# MNIST diffusion speedrun

Generate recognizable, varied digits with fewer denoiser evaluations. Try this optional exercise after pre-class notes 00–02; it uses a separate class-conditional MNIST model. The generator and evaluator are already trained: **no training and no MNIST download are needed to run this notebook**. The notebook checks the released weights and real-data reference before using them.

We use noise at $t=0$, clean data at $t=1$, and the cosine path
$\alpha_t=\sin(\pi t/2)$, $\sigma_t=\cos(\pi t/2)$. The network predicts the clean image from the current noisy image, time, and requested digit.

Install once, from the course repository root:
```sh
python -m pip install -r speedrun/requirements.txt
```
Open this notebook with that Python environment. CUDA, Apple MPS, and CPU are supported. The 100-image preview is quick; the full score uses 10,000 images and is better suited to an accelerator. Timings are measured, not promised across machines.

In [ ]:
from pathlib import Path
import json, os, sys
import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

cwd = Path.cwd().resolve()
ROOT = next((p / "speedrun" for p in (cwd, *cwd.parents)
             if (p / "speedrun/release.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the diffusion-steering-preclass-notes checkout")
sys.path.insert(0, str(ROOT))
import benchmark, sample

torch.set_num_threads(min(4, torch.get_num_threads()))
bench = benchmark.Speedrun(os.environ.get("SPEEDRUN_DEVICE") or None)
print("Device:", bench.device)
print("Release:", bench.manifest["protocol"], "— all three artifact hashes verified")
print("Evaluator test accuracy:", bench.manifest["evaluator_test_accuracy"])
assert not any(p.requires_grad for m in (bench.net, bench.evaluator) for p in m.parameters())

## First, look at 100 generated digits

Each row requests one digit. The input is fresh Gaussian noise plus a label, not a real MNIST image. The denoiser repeatedly estimates a clean image; the sampler updates the current noisy image. This grid is for inspection, **not a 100-sample FID score**.

In [ ]:
def show_grid(images, labels, title):
    rows = [torch.cat(list(images[labels == digit][:10, 0]), dim=1) for digit in range(10)]
    sheet = torch.cat(rows, dim=0).numpy()
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(sheet, cmap="gray", vmin=-1, vmax=1)
    ax.set_yticks(np.arange(10)*28 + 14, labels=np.arange(10))
    ax.set_xticks([])
    ax.set_ylabel("Requested digit")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

preview_labels = torch.arange(10).repeat_interleave(10)
preview, nfe = sample.sample(bench.net, 100, preview_labels, bench.device,
                             steps=4, w=2.0, order="heun", seed=7)
show_grid(preview, preview_labels, f"4-step Heun • {nfe} NFE per image")

## Change the sampling knobs

- `steps`: number of time intervals on the fixed uniform grid.
- `w`: classifier-free guidance, $\hat x_1=u+w(c-u)$. Here $u$ and $c$ are the unconditional and conditional clean-image predictions. `w=0` ignores the requested digit; `w=1` uses the conditional branch alone; `w>1` extrapolates away from the unconditional prediction.
- `order`: `ddim` transports the clean/noise estimates along the schedule; `euler` and `heun` integrate the implied velocity. All finish with a clean-image prediction instead of evaluating a singular velocity at $t=1$.
- `eta`: stochastic DDIM noise, between 0 and 1. Use 0 with Euler or Heun.

One conditional or unconditional prediction costs one NFE. With `w=2`, DDIM/Euler cost `2*steps` and Heun costs `2*(2*steps-1)`. Batching both branches does not halve their cost.

Keep seed, sample count, batch size, weights, and evaluator fixed. Try at most five configurations. A useful comparison is 4-step DDIM versus 4-step Heun; the latter uses more NFE at the same step count. For an equal-cost comparison, compare 7-step DDIM to 4-step Heun (both 14 NFE when `w=2`).

In [ ]:
CONFIGS = [
    dict(steps=4, w=2.0, eta=0.0, order="heun"),
    # dict(steps=7, w=2.0, eta=0.0, order="ddim"),  # equal NFE
    # dict(steps=4, w=1.0, eta=0.0, order="ddim"),  # conditional, no extrapolation
]
N = int(os.environ.get("SPEEDRUN_N", "10000"))
REPEATS = int(os.environ.get("SPEEDRUN_REPEATS", "1"))
SEED = bench.manifest["public_seed"]
assert 1 <= len(CONFIGS) <= 5
print(f"Scoring {N:,} images/configuration; {REPEATS} timed run(s) after warm-up")
print("Use REPEATS=3 for median timing comparisons. N<10,000 is only a smoke test.")

## Run the scorecard

**FID-M** compares the generated feature distribution to all 60,000 MNIST training images, using fixed statistics and the frozen 128-dimensional classifier representation. Lower is better; these numbers are not comparable to published Inception FID. The real MNIST test set scores about 5.58 under this protocol.

**Requested-digit agreement** is the fraction the frozen classifier labels as the digit requested from the generator. For example, ask for a 7, generate an image, and count success if the classifier predicts 7. A score of 99.78% means 9,978 of 10,000 requests matched. The JSON field is named `accuracy`. This measures conditioning, not overall image quality: one repeated recognizable prototype per digit could score 100%. The evaluator’s 99.32% test accuracy is a separate measurement on labeled real MNIST images. **Target confidence** is its mean probability for the requested digit. The nonduplicate fraction measures how many generated samples lack a very close generated neighbor. It is a diversity diagnostic, not proof of distribution coverage: even noise can have few duplicates.

The practice band requires FID-M ≤120, requested-digit agreement ≥98%, and nonduplicate fraction ≥95%, using 10,000 images and the fixed batch size. Among configurations inside the band, lower NFE wins. These are practice thresholds; an instructor must calibrate hidden evaluation seeds before using them for a graded leaderboard.

In [ ]:
rows = []
for config in CONFIGS:
    print("Running", config, flush=True)
    row, generated = bench.run(config, n=N, seed=SEED, repeats=REPEATS)
    rows.append(row)
    print(f"FID-M={row['fid_m']:.2f}, digit agreement={row['accuracy']:.4f}, "
          f"nonduplicates={row['nonduplicate_fraction']:.4f}, NFE={row['nfe_per_sample']}, "
          f"sampling={row['sampling_seconds_median']:.1f}s", flush=True)

out = ROOT / "results/local"
out.mkdir(parents=True, exist_ok=True)
(out / "notebook-scorecards.json").write_text(json.dumps(rows, indent=2) + "\n")
labels = torch.arange(N) % 10
benchmark.save_grid(generated, labels, out / "selected-grid.png")
show_grid(generated, labels, "Last configuration • fixed evaluation seed")

In [ ]:
reference = json.loads((ROOT / "results/reference-seed0.json").read_text())
all_rows = [("Saved reference (recorded device)", reference)] + [(f"Your run {i+1}", row) for i, row in enumerate(rows)]
lines = ["| Run | Sampler | Steps | w | NFE | FID-M | Digit agreement | Nonduplicates | Seconds | In practice band? |",
         "|---|---|---:|---:|---:|---:|---:|---:|---:|---|"]
for name, row in all_rows:
    c = row["config"]
    lines.append(f"| {name} | {c['order']} | {c['steps']} | {c['w']} | {row['nfe_per_sample']} | "
                 f"{row['fid_m']:.2f} | {row['accuracy']:.3f} | {row['nonduplicate_fraction']:.3f} | "
                 f"{row['sampling_seconds_median']:.1f} | {row['meets_practice_band']} |")
display(Markdown("\n".join(lines)))
print("Saved reference device:", reference["device"], "| Your device:", bench.device)
print("Compare timing only on the same device and with three measured repetitions.")
assert benchmark.verify_release(ROOT)["sha256"] == bench.manifest["sha256"]

## Explain your result

1. Which configuration met the practice band using the fewest NFE? Compare equal NFE as well as equal steps.
2. Did stronger guidance improve requested-digit agreement while worsening FID-M or diversity?
3. Include one unsuccessful configuration and explain what failed.
4. Inspect the image grid. Can a configuration pass requested-digit agreement while producing visually repetitive digits?

Submit this executed notebook, up to five complete scorecards, one fixed-seed grid, and a short explanation. Never choose a seed after inspecting its score. The saved reference is a measured example, not a guarantee that every device or seed gives identical numbers.

Implementation: [samplers](sample.py), [frozen scorer](benchmark.py), [metric](fidm.py), [release](release.json). The guidance formula uses the course convention; see [classifier-free guidance](https://arxiv.org/abs/2207.12598). The stochastic DDIM update follows the [authors' implementation](https://github.com/ermongroup/ddim/blob/main/functions/denoising.py).